# Practical 02: Data Hygiene: Missing Value Imputation, Deduplication & Renaming

**UNIT 1: Introduction to Big Data and Large Dataset Handling**

**Problem Statement:** Write a Python program to perform data hygiene on a dataset with missing values and duplicate rows. Identify missing entries using isna().sum(), demonstrate both dropping rows (dropna()) and imputing missing numerical values with fillna(mean/median), remove duplicate records using drop_duplicates(), and rename columns using rename().

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None, "display.width", 200)

rg = np.random.default_rng(99)
n = 170
frame = pd.DataFrame({
    "Order ID": np.arange(1001, 1001 + n),
    "Customer Name": [f"Buyer_{i:03d}" for i in rg.integers(1, 55, n)],
    "City": rg.choice(["Kolkata", "Patna", "Ranchi", "Delhi", "Lucknow", "Surat"], n),
    "Product Category": rg.choice(["Electronics", "Clothing", "Groceries", "Furniture", "Toys"], n),
    "Quantity": rg.integers(1, 10, n).astype(float),
    "Unit Price": np.round(rg.uniform(120, 5500, n), 2),
    "Discount %": rg.choice([0, 5, 10, 15, 20], n).astype(float),
    "Order Date": pd.date_range("2025-04-10", periods=n, freq="D").strftime("%Y-%m-%d"),
})
for col, k in [("Quantity", 11), ("Unit Price", 9), ("Discount %", 10), ("City", 7)]:
    frame.loc[rg.choice(n, k, replace=False), col] = np.nan
frame = pd.concat([frame, frame.sample(14, random_state=4)])
frame = frame.sample(frac=1, random_state=8).reset_index(drop=True)

print("Original shape:", frame.shape)
print(frame.head())

print(frame.isna().sum())

df_no_na = frame.dropna()
print(f"dropna(): {frame.shape[0]} -> {df_no_na.shape[0]} rows ({frame.shape[0] - df_no_na.shape[0]} rows lost)")

df_imp = frame.copy()
df_imp["Quantity"] = df_imp["Quantity"].fillna(df_imp["Quantity"].mean())
df_imp["Unit Price"] = df_imp["Unit Price"].fillna(df_imp["Unit Price"].mean())
df_imp["Discount %"] = df_imp["Discount %"].fillna(df_imp["Discount %"].median())
df_imp["City"] = df_imp["City"].fillna(df_imp["City"].mode()[0])

print("Missing values after imputation:")
print(df_imp.isna().sum())

print("Duplicate rows found:", df_imp.duplicated().sum())
df_tidy = df_imp.drop_duplicates().reset_index(drop=True)
print("Shape after drop_duplicates():", df_tidy.shape)

df_tidy = df_tidy.rename(columns={
    "Order ID": "id",
    "Customer Name": "customer",
    "City": "city",
    "Product Category": "category",
    "Quantity": "qty",
    "Unit Price": "price",
    "Discount %": "discount",
    "Order Date": "date",
})
print(df_tidy.columns.tolist())
print(df_tidy.head())

Original shape: (184, 8)

,Order ID,Customer Name,City,Product Category,Quantity,Unit Price,Discount %,Order Date
0,1157,Buyer_019,Patna,Furniture,3.0,2656.31,20.0,2025-09-13
1,1024,Buyer_014,Surat,Furniture,7.0,2072.76,15.0,2025-05-03
2,1100,Buyer_039,Delhi,Clothing,NaN,750.94,15.0,2025-07-18
3,1135,Buyer_020,Lucknow,Clothing,4.0,257.21,10.0,2025-08-22
4,1002,Buyer_028,Surat,Groceries,7.0,5267.09,10.0,2025-04-11


Order ID             0
Customer Name        0
City                 7
Product Category     0
Quantity            11
Unit Price          10
Discount %          12
Order Date           0
dtype: int64
dropna(): 184 -> 149 rows (35 rows lost)
Missing values after imputation:
Order ID            0
Customer Name       0
City                0
Product Category    0
Quantity            0
Unit Price          0
Discount %          0
Order Date          0
dtype: int64
Duplicate rows found: 14
Shape after drop_duplicates(): (170, 8)
['id', 'customer', 'city', 'category', 'qty', 'price', 'discount', 'date']

,id,customer,city,category,qty,price,discount,date
0,1157,Buyer_019,Patna,Furniture,3.000000,2656.31,20.0,2025-09-13
1,1024,Buyer_014,Surat,Furniture,7.000000,2072.76,15.0,2025-05-03
2,1100,Buyer_039,Delhi,Clothing,5.260116,750.94,15.0,2025-07-18
3,1135,Buyer_020,Lucknow,Clothing,4.000000,257.21,10.0,2025-08-22
4,1002,Buyer_028,Surat,Groceries,7.000000,5267.09,10.0,2025-04-11
